# CAN Rule-Based Detector — Simplified, Split Training/Evaluation

This version keeps the detector rule-based, but trims the feature space to a compact set that is actually consumed by the rules.

Main changes:

- Removed unused byte/nibble/bit columns, binary strings, rolling payload statistics, and evaluation-time transition-probability features.
- Reduced the detector to four interpretable attack families: `DoS`, `fuzzy`, `spoofing`, and `replay`.
- Added a persisted detector profile (`can_detector_profile.json`) containing global thresholds, per-ID thresholds, allowed IDs, allowed DLCs, and learned ID transitions.
- Split execution into a training stage and an evaluation stage. Train once, save the profile, then rerun only the evaluation cell with different files.


In [50]:
from __future__ import annotations

import json
import re
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence, Set, Tuple

import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)


## User Configuration


In [51]:
# ============================================================
# USER CONFIGURATION
# ============================================================

# Benign-only training dataset used once to calibrate normal behavior.
TRAINING_FILE = "training.csv"

# Saved profile created by the training stage and reused by the evaluation stage.
DETECTOR_PROFILE_FILE = "can_detector_profile.json"

# One or more evaluation datasets. Change these and rerun only the evaluation cell.
EVALUATION_FILES = [
    "eval3.csv",
]

# Evaluation mode: "binary" for normal vs attack, or "multiclass" for attack family names.
EVALUATION_MODE = "binary"

# Set this to True only when you want to create/update DETECTOR_PROFILE_FILE.
# Leave it False when rerunning evaluation with different files.
RUN_TRAINING_STAGE = False
RUN_EVALUATION_STAGE = True

# Column mapping. Edit these if your CSV uses different column names.
TIMESTAMP_COL = "timestamp"
ARBITRATION_ID_COL = "arbitration_id"
DATA_FIELD_COL = "data_field"
LABEL_COL = "attack"  # Set to None if your evaluation data has no true labels.
BYTE_COLS = None      # Example: ["byte_0", "byte_1", ..., "byte_7"]

# Feature extraction settings. These are saved in the detector profile at training time.
MAX_PAYLOAD_BYTES = 8
ROLLING_PACKET_WINDOW = 100
TIME_WINDOW_SECONDS = 1.0

# Rule calibration settings.
HIGH_QUANTILE = 0.995
LOW_QUANTILE = 0.005
MIN_ATTACK_SCORE = 2.0


## Constants and Configuration


In [52]:
# ============================================================
# GLOBAL CONSTANTS AND CONFIGURATION
# ============================================================

CAN_CLASSIC_MAX_BYTES = 8
CAN_FD_MAX_BYTES = 64
STANDARD_CAN_MAX_ID = 0x7FF
EXTENDED_CAN_MAX_ID = 0x1FFFFFFF
HEX_PATTERN = re.compile(r"^[0-9A-F]*$")


@dataclass
class CANColumnConfig:
    """
    Central place to adapt the notebook to different datasets.
    """

    timestamp_col: str = "timestamp"
    arbitration_id_col: str = "arbitration_id"
    data_field_col: str = "data_field"
    label_col: Optional[str] = "attack"
    byte_cols: Optional[List[str]] = None


def build_column_config() -> CANColumnConfig:
    return CANColumnConfig(
        timestamp_col=TIMESTAMP_COL,
        arbitration_id_col=ARBITRATION_ID_COL,
        data_field_col=DATA_FIELD_COL,
        label_col=LABEL_COL,
        byte_cols=BYTE_COLS,
    )


# Printed output settings.
PRINT_TOP_SUSPICIOUS_ROWS = 20
PRINT_FULL_THRESHOLD_TABLE = True
PRINT_PREDICTION_COLUMNS = [
    "timestamp_seconds",
    "arbitration_id",
    "arbitration_id_hex",
    "dlc",
    "payload_hex_normalized",
    "predicted_label",
    "max_rule_score",
]

CORE_FEATURE_COLUMNS = [
    "delta_t_global_s",
    "delta_t_same_id_s",
    "packet_rate_last_time_window_hz",
    "same_id_rate_last_time_window_hz",
    "consecutive_id_streak_length",
    "prev_arbitration_id",
    "rolling_unique_id_count",
    "rolling_id_entropy_bits",
    "payload_byte_sum",
    "payload_byte_entropy_bits",
    "payload_hamming_distance_prev_same_id",
]


## CSV Loading and Column Standardization


In [53]:
# ============================================================
# CSV LOADING AND COLUMN STANDARDIZATION
# ============================================================

def load_can_csv(
    path: str | Path,
    *,
    has_header: bool = True,
    columns: Tuple[str, str, str, str] = (
        "timestamp",
        "arbitration_id",
        "data_field",
        "attack",
    ),
) -> pd.DataFrame:
    """
    Load a CAN CSV while preserving hexadecimal payload strings.
    """
    path = Path(path)

    if has_header:
        return pd.read_csv(path, dtype="string")

    return pd.read_csv(
        path,
        names=list(columns),
        header=None,
        dtype="string",
    )


def _build_data_field_from_byte_columns(
    raw_df: pd.DataFrame,
    byte_cols: Sequence[str],
) -> pd.Series:
    """
    Convert separate byte columns into one hexadecimal data_field string.
    """

    def row_to_hex(row: pd.Series) -> str:
        payload_parts: List[str] = []

        for col in byte_cols:
            value = row.get(col, np.nan)

            if pd.isna(value):
                continue

            try:
                byte_value = int(value)
            except (TypeError, ValueError):
                try:
                    text = str(value).strip().upper()
                    if text.startswith("0X"):
                        text = text[2:]
                    byte_value = int(text, 16)
                except (TypeError, ValueError):
                    continue

            if 0 <= byte_value <= 255:
                payload_parts.append(f"{byte_value:02X}")

        return "".join(payload_parts)

    return raw_df.apply(row_to_hex, axis=1)


def standardize_raw_can_columns(
    raw_df: pd.DataFrame,
    config: CANColumnConfig,
) -> pd.DataFrame:
    """
    Convert different dataset schemas into the canonical schema expected by
    preprocess_can_dataframe():

        timestamp
        arbitration_id
        data_field
        attack

    The true label is copied to __true_label and is never used for feature
    extraction or rule calibration.
    """
    df = raw_df.copy()

    required = [config.timestamp_col, config.arbitration_id_col]
    missing = [col for col in required if col not in df.columns]
    if missing:
        raise ValueError(f"Missing required input columns: {missing}")

    standardized = pd.DataFrame(index=df.index)
    standardized["timestamp"] = df[config.timestamp_col]
    standardized["arbitration_id"] = df[config.arbitration_id_col]

    if config.data_field_col in df.columns:
        standardized["data_field"] = df[config.data_field_col]
    elif config.byte_cols is not None:
        missing_bytes = [col for col in config.byte_cols if col not in df.columns]
        if missing_bytes:
            raise ValueError(f"Missing byte columns: {missing_bytes}")
        standardized["data_field"] = _build_data_field_from_byte_columns(df, config.byte_cols)
    else:
        raise ValueError("No data_field column found and no byte_cols were provided.")

    if config.label_col is not None and config.label_col in df.columns:
        standardized["__true_label"] = df[config.label_col].values

    # Added for compatibility with preprocessing; it is dropped immediately.
    standardized["attack"] = 0

    return standardized


## Preprocessing


In [54]:
# ============================================================
# PREPROCESSING HELPERS
# ============================================================

def _parse_can_identifier(value: object) -> float:
    """
    Parse arbitration IDs robustly.

    Rules:
      - 0x-prefixed values are parsed as hexadecimal.
      - Values containing A-F are parsed as hexadecimal.
      - Digit-only values are parsed as decimal.
      - Invalid or out-of-range values become NaN.
    """
    if pd.isna(value):
        return np.nan

    text = str(value).strip().upper()
    if not text:
        return np.nan

    try:
        if text.startswith("0X"):
            parsed = int(text[2:], 16)
        elif any(ch in text for ch in "ABCDEF"):
            parsed = int(text, 16)
        else:
            parsed = int(text, 10)
    except (TypeError, ValueError):
        return np.nan

    if parsed < 0 or parsed > EXTENDED_CAN_MAX_ID:
        return np.nan

    return float(parsed)


def _normalize_payload_hex(value: object) -> Tuple[str, bool, bool, bool]:
    """
    Normalize CAN data_field into uppercase hexadecimal text.

    Returns:
      normalized_hex, is_valid_hex, is_missing, has_odd_nibble_count
    """
    if pd.isna(value):
        return "", False, True, False

    text = str(value).strip().upper()

    if text.startswith("0X"):
        text = text[2:]

    # Remove common separators from logs.
    for sep in (" ", "-", ":", "_", ","):
        text = text.replace(sep, "")

    if text == "":
        return "", False, True, False

    contains_only_hex = bool(HEX_PATTERN.fullmatch(text))
    odd_nibbles = len(text) % 2 == 1
    is_valid_hex = contains_only_hex and not odd_nibbles

    return text, is_valid_hex, False, odd_nibbles


# ============================================================
# PREPROCESSING
# ============================================================

def preprocess_can_dataframe(
    raw_df: pd.DataFrame,
    *,
    max_payload_bytes: int = CAN_CLASSIC_MAX_BYTES,
) -> pd.DataFrame:
    """
    Convert raw CAN packets into a compact, rule-ready representation.

    The earlier notebook unfolded many byte, nibble, bit, and binary-string
    columns. This simplified version keeps only fields needed for rules,
    diagnostics, and final display.
    """
    if max_payload_bytes < 1 or max_payload_bytes > CAN_FD_MAX_BYTES:
        raise ValueError("max_payload_bytes must be between 1 and 64.")

    df = raw_df.copy()

    # Drop attack-like column immediately. Evaluation labels should be stored
    # in __true_label before this point.
    if "attack" in df.columns:
        df = df.drop(columns=["attack"])

    required_columns = {"timestamp", "arbitration_id", "data_field"}
    missing = required_columns.difference(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")

    # Timestamp processing.
    df["timestamp_raw"] = df["timestamp"]
    df["timestamp_seconds"] = pd.to_numeric(df["timestamp"], errors="coerce")
    df["timestamp_parse_valid"] = df["timestamp_seconds"].notna()

    # Stable chronological sort keeps equal timestamps in original order.
    df = df.sort_values(
        "timestamp_seconds",
        kind="mergesort",
        na_position="last",
    ).reset_index(drop=True)

    first_valid_ts = df["timestamp_seconds"].dropna().min()
    if pd.isna(first_valid_ts):
        first_valid_ts = 0.0
    df["time_from_start_s"] = df["timestamp_seconds"] - first_valid_ts

    # Arbitration ID processing.
    df["arbitration_id_raw"] = df["arbitration_id"]
    df["arbitration_id"] = df["arbitration_id"].map(_parse_can_identifier)
    df["arbitration_id_parse_valid"] = df["arbitration_id"].notna()
    df["arbitration_id_hex"] = df["arbitration_id"].map(
        lambda x: f"0x{int(x):X}" if pd.notna(x) else pd.NA
    )

    # Payload processing.
    df["data_field_raw"] = df["data_field"]
    payload_parts = pd.DataFrame(
        df["data_field"].map(_normalize_payload_hex).tolist(),
        columns=[
            "payload_hex_normalized",
            "payload_is_valid_hex",
            "payload_is_missing",
            "payload_has_odd_nibble_count",
        ],
        index=df.index,
    )
    df = pd.concat([df, payload_parts], axis=1)

    dlc = np.where(
        df["payload_is_valid_hex"],
        df["payload_hex_normalized"].str.len() // 2,
        np.nan,
    )
    df["dlc"] = pd.Series(dlc, index=df.index, dtype="float64").astype("Int64")
    df["payload_exceeds_configured_length"] = df["dlc"].fillna(0) > max_payload_bytes

    # Preserve parsed columns; remove raw ambiguous working inputs.
    df = df.drop(columns=["timestamp", "data_field"])

    return df


## Compact Feature Extraction


In [55]:
# ============================================================
# COMPACT FEATURE EXTRACTION HELPERS
# ============================================================

def _safe_entropy_from_values(values: Iterable[Any]) -> float:
    """
    Shannon entropy in bits for finite values.
    """
    arr = pd.Series(list(values)).dropna().to_numpy()

    if arr.size == 0:
        return np.nan

    _, counts = np.unique(arr, return_counts=True)
    probabilities = counts / counts.sum()
    probabilities = probabilities[probabilities > 0]

    return float(-(probabilities * np.log2(probabilities)).sum())


def _rolling_entropy_numeric(
    series: pd.Series,
    window: int,
    min_periods: int,
) -> pd.Series:
    return series.rolling(window=window, min_periods=min_periods).apply(
        _safe_entropy_from_values,
        raw=False,
    )


def _count_in_previous_seconds(
    timestamps: np.ndarray,
    window_seconds: float,
) -> np.ndarray:
    """
    Count packets in [t - window_seconds, t].
    """
    out = np.full(len(timestamps), np.nan, dtype="float64")
    valid_positions = np.flatnonzero(np.isfinite(timestamps))

    if valid_positions.size == 0:
        return out

    valid_t = timestamps[valid_positions]
    left = np.searchsorted(valid_t, valid_t - window_seconds, side="left")
    out[valid_positions] = np.arange(valid_t.size) - left + 1

    return out


def _group_count_in_previous_seconds(
    df: pd.DataFrame,
    id_col: str,
    time_col: str,
    window_seconds: float,
) -> np.ndarray:
    out = np.full(len(df), np.nan, dtype="float64")
    timestamps = df[time_col].to_numpy(dtype="float64")

    for _, index_values in df.groupby(id_col, sort=False, dropna=True).indices.items():
        positions = np.asarray(index_values, dtype=int)
        group_t = timestamps[positions]
        out[positions] = _count_in_previous_seconds(group_t, window_seconds)

    return out


def _payload_byte_matrix(
    payload_hex: pd.Series,
    valid_hex: pd.Series,
    max_payload_bytes: int,
) -> np.ndarray:
    """
    Convert normalized payload text into a fixed-width numeric byte matrix.
    Values not present or invalid are NaN.
    """
    matrix = np.full((len(payload_hex), max_payload_bytes), np.nan, dtype="float64")

    for row_idx, (hex_text, is_valid) in enumerate(zip(payload_hex, valid_hex)):
        if not bool(is_valid):
            continue

        text = str(hex_text)[: max_payload_bytes * 2]
        try:
            payload = bytes.fromhex(text)
        except ValueError:
            continue

        if payload:
            usable = payload[:max_payload_bytes]
            matrix[row_idx, : len(usable)] = list(usable)

    return matrix


def _previous_same_id_matrix(
    df: pd.DataFrame,
    id_col: str,
    matrix: np.ndarray,
) -> np.ndarray:
    """
    For each packet, return the previous payload byte row with the same CAN ID.
    """
    previous = np.full_like(matrix, np.nan, dtype="float64")

    for _, index_values in df.groupby(id_col, sort=False, dropna=True).indices.items():
        positions = np.asarray(index_values, dtype=int)
        if len(positions) > 1:
            previous[positions[1:], :] = matrix[positions[:-1], :]

    return previous


# ============================================================
# COMPACT FEATURE EXTRACTION
# ============================================================

def extract_can_features(
    preprocessed_df: pd.DataFrame,
    *,
    rolling_packet_window: int = 100,
    time_window_seconds: float = 1.0,
    max_payload_bytes: int = CAN_CLASSIC_MAX_BYTES,
    print_summary: bool = True,
) -> Tuple[pd.DataFrame, List[str]]:
    """
    Extract only the features used by the simplified detector.
    """
    if rolling_packet_window < 2:
        raise ValueError("rolling_packet_window must be >= 2.")
    if time_window_seconds <= 0:
        raise ValueError("time_window_seconds must be positive.")
    if max_payload_bytes < 1 or max_payload_bytes > CAN_FD_MAX_BYTES:
        raise ValueError("max_payload_bytes must be between 1 and 64.")

    df = preprocessed_df.copy()
    generated_features: List[str] = []

    def add_feature(name: str, values: Any) -> None:
        df[name] = values
        generated_features.append(name)

    n = len(df)
    t = df["timestamp_seconds"].astype("float64")
    id_series = df["arbitration_id"].astype("float64")

    # Time and rate features.
    add_feature("delta_t_global_s", t.diff())
    add_feature(
        "delta_t_same_id_s",
        df.groupby("arbitration_id", dropna=True)["timestamp_seconds"].diff(),
    )

    packet_count = _count_in_previous_seconds(t.to_numpy(dtype="float64"), time_window_seconds)
    add_feature("packet_rate_last_time_window_hz", packet_count / time_window_seconds)

    same_id_count = _group_count_in_previous_seconds(
        df,
        "arbitration_id",
        "timestamp_seconds",
        time_window_seconds,
    )
    add_feature("same_id_rate_last_time_window_hz", same_id_count / time_window_seconds)

    streak_group = id_series.ne(id_series.shift(1)).cumsum()
    add_feature("consecutive_id_streak_length", df.groupby(streak_group).cumcount() + 1)
    add_feature("prev_arbitration_id", id_series.shift(1))

    # Rolling ID diversity features: useful for fuzzy/random-ID traffic.
    id_codes = pd.Series(pd.factorize(df["arbitration_id"], sort=False)[0], index=df.index).replace(-1, np.nan)
    add_feature(
        "rolling_unique_id_count",
        id_codes.rolling(rolling_packet_window, min_periods=2).apply(
            lambda x: len(pd.unique(pd.Series(x).dropna())),
            raw=False,
        ),
    )
    add_feature(
        "rolling_id_entropy_bits",
        _rolling_entropy_numeric(id_codes, rolling_packet_window, min_periods=2),
    )

    # Payload features.
    byte_matrix_float = _payload_byte_matrix(
        df["payload_hex_normalized"],
        df["payload_is_valid_hex"],
        max_payload_bytes,
    )
    byte_valid = np.isfinite(byte_matrix_float)
    byte_matrix_uint = np.nan_to_num(byte_matrix_float, nan=0).astype(np.uint8)
    valid_byte_count = byte_valid.sum(axis=1)

    payload_byte_sum = np.where(
        valid_byte_count > 0,
        np.where(byte_valid, byte_matrix_float, 0).sum(axis=1),
        np.nan,
    )
    add_feature("payload_byte_sum", payload_byte_sum)

    payload_byte_entropy = np.array(
        [_safe_entropy_from_values(row[valid]) for row, valid in zip(byte_matrix_uint, byte_valid)],
        dtype="float64",
    )
    add_feature("payload_byte_entropy_bits", payload_byte_entropy)

    bit_count_lut = np.unpackbits(
        np.arange(256, dtype=np.uint8)[:, None],
        axis=1,
    ).sum(axis=1)

    prev_same_id_bytes = _previous_same_id_matrix(df, "arbitration_id", byte_matrix_float)
    prev_same_id_valid = np.isfinite(prev_same_id_bytes)
    prev_same_id_uint = np.nan_to_num(prev_same_id_bytes, nan=0).astype(np.uint8)

    comparable_same_id = byte_valid & prev_same_id_valid
    xor_same_id = np.bitwise_xor(byte_matrix_uint, prev_same_id_uint)
    hamming_same_id = (bit_count_lut[xor_same_id] * comparable_same_id).sum(axis=1).astype("float64")
    hamming_same_id[comparable_same_id.sum(axis=1) == 0] = np.nan
    add_feature("payload_hamming_distance_prev_same_id", hamming_same_id)

    generated_features = list(dict.fromkeys(generated_features))
    df[generated_features] = df[generated_features].replace([np.inf, -np.inf], np.nan)

    if print_summary:
        print(f"Total generated features: {len(generated_features)}")
        print("\nGenerated core features:")
        print(pd.Series(generated_features).to_string(index=False))
        print("\nExample rows:")
        preview_cols = [
            "timestamp_seconds",
            "arbitration_id_hex",
            "dlc",
            "payload_hex_normalized",
        ] + generated_features
        print(df[[col for col in preview_cols if col in df.columns]].head().to_string(index=False))

    return df, generated_features


## End-to-End Feature Wrapper


In [56]:
# ============================================================
# END-TO-END FEATURE EXTRACTION WRAPPER
# ============================================================

def extract_features_for_rule_detector(
    raw_df: pd.DataFrame,
    config: CANColumnConfig,
    *,
    max_payload_bytes: int = 8,
    rolling_packet_window: int = 100,
    time_window_seconds: float = 1.0,
    print_summary: bool = False,
) -> Tuple[pd.DataFrame, Optional[pd.Series]]:
    """
    Standardize columns, preprocess packets, and extract the compact feature set.
    """
    standardized = standardize_raw_can_columns(raw_df, config)
    has_true_label = "__true_label" in standardized.columns

    preprocessed_df = preprocess_can_dataframe(
        standardized,
        max_payload_bytes=max_payload_bytes,
    )

    features_df, _ = extract_can_features(
        preprocessed_df,
        rolling_packet_window=rolling_packet_window,
        time_window_seconds=time_window_seconds,
        max_payload_bytes=max_payload_bytes,
        print_summary=print_summary,
    )

    true_labels = None
    if has_true_label and "__true_label" in features_df.columns:
        true_labels = features_df["__true_label"].copy()
        features_df = features_df.drop(columns=["__true_label"])

    return features_df, true_labels


## Simplified Rule-Based Detector with Saved Profile


In [57]:
# ============================================================
# SIMPLIFIED RULE-BASED DETECTOR
# ============================================================

class CANRuleBasedDetector:
    """
    Compact rule-based CAN attack detector calibrated from benign traffic.

    The saved profile includes:
      - global thresholds
      - known benign CAN IDs
      - allowed DLC values per ID
      - per-ID timing, rate, hamming, and payload-sum thresholds
      - allowed ID transitions learned from benign training traffic
    """

    profile_version = "simplified-v2"

    def __init__(
        self,
        *,
        high_quantile: float = 0.995,
        low_quantile: float = 0.005,
        min_attack_score: float = 2.0,
    ) -> None:
        self.high_quantile = high_quantile
        self.low_quantile = low_quantile
        self.min_attack_score = min_attack_score

        self.global_thresholds: Dict[str, float] = {}
        self.allowed_ids: Set[float] = set()
        self.allowed_dlc_by_id: Dict[float, Set[int]] = {}
        self.allowed_transitions: Set[Tuple[float, float]] = set()

        self.id_delta_t_low: Dict[float, float] = {}
        self.id_delta_t_high: Dict[float, float] = {}
        self.id_rate_high: Dict[float, float] = {}
        self.id_hamming_high: Dict[float, float] = {}
        self.id_payload_sum_low: Dict[float, float] = {}
        self.id_payload_sum_high: Dict[float, float] = {}

        self.feature_config: Dict[str, Any] = {}
        self.profile_metadata: Dict[str, Any] = {}

    @staticmethod
    def _safe_quantile(series: pd.Series, q: float, default: float) -> float:
        values = pd.to_numeric(series, errors="coerce")
        values = values.replace([np.inf, -np.inf], np.nan).dropna()
        if values.empty:
            return default
        return float(values.quantile(q))

    @staticmethod
    def _id_key(can_id: float) -> str:
        return str(int(float(can_id)))

    @staticmethod
    def _json_number(value: Any) -> Any:
        if value is None:
            return None
        try:
            value = float(value)
        except (TypeError, ValueError):
            return None
        if np.isnan(value):
            return None
        if np.isposinf(value):
            return "inf"
        if np.isneginf(value):
            return "-inf"
        return value

    @staticmethod
    def _from_json_number(value: Any, default: float = np.nan) -> float:
        if value is None:
            return default
        if value == "inf":
            return np.inf
        if value == "-inf":
            return -np.inf
        return float(value)

    def fit(self, benign_features_df: pd.DataFrame) -> "CANRuleBasedDetector":
        """
        Calibrate normal thresholds from benign-only training features.
        """
        df = benign_features_df.copy()

        required = {
            "arbitration_id",
            "dlc",
            "delta_t_global_s",
            "delta_t_same_id_s",
            "packet_rate_last_time_window_hz",
            "same_id_rate_last_time_window_hz",
            "consecutive_id_streak_length",
            "prev_arbitration_id",
            "rolling_unique_id_count",
            "rolling_id_entropy_bits",
            "payload_byte_sum",
            "payload_byte_entropy_bits",
            "payload_hamming_distance_prev_same_id",
        }
        missing = sorted(required.difference(df.columns))
        if missing:
            raise ValueError(f"Cannot fit detector; missing feature columns: {missing}")

        self.allowed_ids = set(
            pd.to_numeric(df["arbitration_id"], errors="coerce")
            .dropna()
            .astype(float)
            .unique()
        )

        self.global_thresholds = {
            "delta_t_global_low": self._safe_quantile(df["delta_t_global_s"], self.low_quantile, 0.0),
            "packet_rate_high": self._safe_quantile(df["packet_rate_last_time_window_hz"], self.high_quantile, np.inf),
            "same_id_rate_high": self._safe_quantile(df["same_id_rate_last_time_window_hz"], self.high_quantile, np.inf),
            "consecutive_id_streak_high": max(
                5.0,
                self._safe_quantile(df["consecutive_id_streak_length"], self.high_quantile, 5.0),
            ),
            "rolling_unique_id_count_high": self._safe_quantile(df["rolling_unique_id_count"], self.high_quantile, np.inf),
            "rolling_id_entropy_high": self._safe_quantile(df["rolling_id_entropy_bits"], self.high_quantile, np.inf),
            "payload_entropy_high": self._safe_quantile(df["payload_byte_entropy_bits"], self.high_quantile, np.inf),
            "hamming_same_id_high": self._safe_quantile(df["payload_hamming_distance_prev_same_id"], self.high_quantile, np.inf),
        }

        for arbitration_id, group in df.groupby("arbitration_id", dropna=True):
            arbitration_id = float(arbitration_id)
            valid_dlc = pd.to_numeric(group["dlc"], errors="coerce").dropna().astype(int).unique()
            self.allowed_dlc_by_id[arbitration_id] = set(valid_dlc)
            self.id_delta_t_low[arbitration_id] = self._safe_quantile(group["delta_t_same_id_s"], self.low_quantile, 0.0)
            self.id_delta_t_high[arbitration_id] = self._safe_quantile(group["delta_t_same_id_s"], self.high_quantile, np.inf)
            self.id_rate_high[arbitration_id] = self._safe_quantile(group["same_id_rate_last_time_window_hz"], self.high_quantile, np.inf)
            self.id_hamming_high[arbitration_id] = self._safe_quantile(group["payload_hamming_distance_prev_same_id"], self.high_quantile, np.inf)
            self.id_payload_sum_low[arbitration_id] = self._safe_quantile(group["payload_byte_sum"], self.low_quantile, -np.inf)
            self.id_payload_sum_high[arbitration_id] = self._safe_quantile(group["payload_byte_sum"], self.high_quantile, np.inf)

        transition_base = df.dropna(subset=["prev_arbitration_id", "arbitration_id"])
        self.allowed_transitions = set(
            zip(
                pd.to_numeric(transition_base["prev_arbitration_id"], errors="coerce").astype(float),
                pd.to_numeric(transition_base["arbitration_id"], errors="coerce").astype(float),
            )
        )

        return self

    def save(self, path: str | Path, *, feature_config: Optional[Dict[str, Any]] = None) -> None:
        """
        Save calibrated thresholds and learned benign profiles to JSON.
        """
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)

        self.feature_config = dict(feature_config or self.feature_config or {})
        self.profile_metadata = {
            "created_at_utc": datetime.now(timezone.utc).isoformat(),
            "profile_version": self.profile_version,
        }

        all_ids = sorted(self.allowed_ids)
        id_thresholds: Dict[str, Dict[str, Any]] = {}
        for can_id in all_ids:
            key = self._id_key(can_id)
            id_thresholds[key] = {
                "delta_t_low": self._json_number(self.id_delta_t_low.get(can_id, 0.0)),
                "delta_t_high": self._json_number(self.id_delta_t_high.get(can_id, np.inf)),
                "same_id_rate_high": self._json_number(self.id_rate_high.get(can_id, np.inf)),
                "hamming_high": self._json_number(self.id_hamming_high.get(can_id, np.inf)),
                "payload_sum_low": self._json_number(self.id_payload_sum_low.get(can_id, -np.inf)),
                "payload_sum_high": self._json_number(self.id_payload_sum_high.get(can_id, np.inf)),
            }

        profile = {
            "metadata": self.profile_metadata,
            "high_quantile": self.high_quantile,
            "low_quantile": self.low_quantile,
            "min_attack_score": self.min_attack_score,
            "feature_config": self.feature_config,
            "global_thresholds": {
                key: self._json_number(value)
                for key, value in self.global_thresholds.items()
            },
            "allowed_ids": [int(can_id) for can_id in all_ids],
            "allowed_dlc_by_id": {
                self._id_key(can_id): sorted(int(v) for v in values)
                for can_id, values in self.allowed_dlc_by_id.items()
            },
            "allowed_transitions": [
                [int(prev_id), int(current_id)]
                for prev_id, current_id in sorted(self.allowed_transitions)
            ],
            "id_thresholds": id_thresholds,
        }

        with path.open("w", encoding="utf-8") as f:
            json.dump(profile, f, indent=2, sort_keys=True)

    @classmethod
    def load(cls, path: str | Path) -> "CANRuleBasedDetector":
        """
        Load a saved detector profile. No training data is required.
        """
        path = Path(path)
        with path.open("r", encoding="utf-8") as f:
            profile = json.load(f)

        detector = cls(
            high_quantile=float(profile.get("high_quantile", 0.995)),
            low_quantile=float(profile.get("low_quantile", 0.005)),
            min_attack_score=float(profile.get("min_attack_score", 2.0)),
        )
        detector.profile_metadata = dict(profile.get("metadata", {}))
        detector.feature_config = dict(profile.get("feature_config", {}))
        detector.global_thresholds = {
            key: detector._from_json_number(value, default=np.nan)
            for key, value in profile.get("global_thresholds", {}).items()
        }
        detector.allowed_ids = {float(can_id) for can_id in profile.get("allowed_ids", [])}
        detector.allowed_dlc_by_id = {
            float(can_id): set(int(v) for v in values)
            for can_id, values in profile.get("allowed_dlc_by_id", {}).items()
        }
        detector.allowed_transitions = {
            (float(prev_id), float(current_id))
            for prev_id, current_id in profile.get("allowed_transitions", [])
        }

        id_thresholds = profile.get("id_thresholds", {})
        for can_id_text, thresholds in id_thresholds.items():
            can_id = float(can_id_text)
            detector.id_delta_t_low[can_id] = detector._from_json_number(thresholds.get("delta_t_low"), 0.0)
            detector.id_delta_t_high[can_id] = detector._from_json_number(thresholds.get("delta_t_high"), np.inf)
            detector.id_rate_high[can_id] = detector._from_json_number(thresholds.get("same_id_rate_high"), np.inf)
            detector.id_hamming_high[can_id] = detector._from_json_number(thresholds.get("hamming_high"), np.inf)
            detector.id_payload_sum_low[can_id] = detector._from_json_number(thresholds.get("payload_sum_low"), -np.inf)
            detector.id_payload_sum_high[can_id] = detector._from_json_number(thresholds.get("payload_sum_high"), np.inf)

        return detector

    def _add_rule(
        self,
        scores: pd.DataFrame,
        flags: pd.DataFrame,
        attack_type: str,
        rule_name: str,
        condition: pd.Series,
        points: float,
    ) -> None:
        condition = condition.fillna(False).astype(bool)
        scores.loc[condition, attack_type] += points
        flags[f"rule_{attack_type}_{rule_name}"] = condition.astype(int)

    def predict(self, features_df: pd.DataFrame, *, return_scores: bool = True) -> pd.DataFrame:
        """
        Apply the simplified explicit rules and return labels, scores, and rule flags.
        """
        df = features_df.copy()

        attack_classes = ["DoS", "fuzzy", "spoofing", "replay"]
        scores = pd.DataFrame(0.0, index=df.index, columns=attack_classes)
        flags = pd.DataFrame(index=df.index)

        arbitration_id = pd.to_numeric(df["arbitration_id"], errors="coerce")
        previous_id = pd.to_numeric(df["prev_arbitration_id"], errors="coerce")
        dlc = pd.to_numeric(df["dlc"], errors="coerce")
        id_known = arbitration_id.isin(self.allowed_ids)
        prev_id_known = previous_id.isin(self.allowed_ids)

        id_delta_low = arbitration_id.map(self.id_delta_t_low)
        id_delta_high = arbitration_id.map(self.id_delta_t_high)
        id_rate_high = arbitration_id.map(self.id_rate_high)
        id_hamming_high = arbitration_id.map(self.id_hamming_high)
        id_payload_sum_low = arbitration_id.map(self.id_payload_sum_low)
        id_payload_sum_high = arbitration_id.map(self.id_payload_sum_high)

        allowed_dlc_pairs = {
            (float(can_id), int(dlc_value))
            for can_id, dlc_values in self.allowed_dlc_by_id.items()
            for dlc_value in dlc_values
        }
        dlc_allowed = pd.Series(
            [
                (float(can_id), int(dlc_value)) in allowed_dlc_pairs
                if pd.notna(can_id) and pd.notna(dlc_value)
                else False
                for can_id, dlc_value in zip(arbitration_id, dlc)
            ],
            index=df.index,
        )
        dlc_unusual_for_known_id = id_known & ~dlc_allowed

        transition_allowed = pd.Series(
            [
                (float(prev_id), float(current_id)) in self.allowed_transitions
                if pd.notna(prev_id) and pd.notna(current_id)
                else True
                for prev_id, current_id in zip(previous_id, arbitration_id)
            ],
            index=df.index,
        )
        unexpected_transition = id_known & prev_id_known & ~transition_allowed
        invalid_payload = ~df["payload_is_valid_hex"].astype(bool)

        gt = self.global_thresholds
        same_id_timing_out = id_known & (
            (df["delta_t_same_id_s"] < id_delta_low)
            | (df["delta_t_same_id_s"] > id_delta_high)
        )
        hamming_out = id_known & (
            (df["payload_hamming_distance_prev_same_id"] > id_hamming_high)
            | (df["payload_hamming_distance_prev_same_id"] > gt.get("hamming_same_id_high", np.inf))
        )
        payload_sum_out = id_known & (
            (df["payload_byte_sum"] < id_payload_sum_low)
            | (df["payload_byte_sum"] > id_payload_sum_high)
        )

        # -----------------------------
        # DoS / flooding rules
        # -----------------------------
        self._add_rule(scores, flags, "DoS", "high_global_packet_rate", df["packet_rate_last_time_window_hz"] > gt.get("packet_rate_high", np.inf), 2.0)
        self._add_rule(scores, flags, "DoS", "very_small_global_delta_t", df["delta_t_global_s"] < gt.get("delta_t_global_low", 0.0), 1.5)
        self._add_rule(scores, flags, "DoS", "same_id_rate_too_high", (df["same_id_rate_last_time_window_hz"] > gt.get("same_id_rate_high", np.inf)) | (df["same_id_rate_last_time_window_hz"] > id_rate_high), 2.0)
        self._add_rule(scores, flags, "DoS", "long_consecutive_id_streak", df["consecutive_id_streak_length"] > gt.get("consecutive_id_streak_high", np.inf), 1.5)

        # -----------------------------
        # Fuzzy / malformed traffic rules
        # -----------------------------
        self._add_rule(scores, flags, "fuzzy", "unknown_arbitration_id", ~id_known, 3.0)
        self._add_rule(scores, flags, "fuzzy", "invalid_payload", invalid_payload, 3.0)
        self._add_rule(scores, flags, "fuzzy", "high_id_diversity", (df["rolling_unique_id_count"] > gt.get("rolling_unique_id_count_high", np.inf)) | (df["rolling_id_entropy_bits"] > gt.get("rolling_id_entropy_high", np.inf)), 1.5)
        self._add_rule(scores, flags, "fuzzy", "high_payload_entropy", df["payload_byte_entropy_bits"] > gt.get("payload_entropy_high", np.inf), 1.0)

        # -----------------------------
        # Spoofing / profile-deviation rules
        # -----------------------------
        self._add_rule(scores, flags, "spoofing", "dlc_unusual_for_known_id", dlc_unusual_for_known_id, 2.0)
        self._add_rule(scores, flags, "spoofing", "same_id_timing_out_of_profile", same_id_timing_out, 2.0)
        self._add_rule(scores, flags, "spoofing", "payload_sum_out_of_profile", payload_sum_out, 1.5)
        self._add_rule(scores, flags, "spoofing", "hamming_distance_out_of_profile", hamming_out, 1.5)
        self._add_rule(scores, flags, "spoofing", "unexpected_id_transition", unexpected_transition, 1.5)

        # -----------------------------
        # Replay-like rule
        # -----------------------------
        repeated_payload = df["payload_hamming_distance_prev_same_id"] == 0
        self._add_rule(scores, flags, "replay", "repeated_payload_bad_timing", id_known & repeated_payload & same_id_timing_out, 2.0)

        max_score = scores.max(axis=1)
        predicted_label = scores.idxmax(axis=1)
        predicted_label[max_score < self.min_attack_score] = "normal"

        output = df.copy()
        output["predicted_label"] = predicted_label
        output["max_rule_score"] = max_score

        if return_scores:
            for col in scores.columns:
                output[f"score_{col}"] = scores[col]
            output = pd.concat([output, flags], axis=1)

        return output


## Evaluation Helpers


In [58]:
# ============================================================
# EVALUATION HELPERS
# ============================================================

def normalize_binary_label(value: Any) -> str:
    """
    Convert labels into normal / attack.
    """
    if pd.isna(value):
        return "normal"

    text = str(value).strip().lower()
    normal_values = {"0", "normal", "benign", "false", "none", "no_attack", "no attack"}
    if text in normal_values:
        return "normal"
    return "attack"


def normalize_multiclass_label(value: Any) -> str:
    """
    Convert dataset-specific labels into the simplified attack family names.
    """
    if pd.isna(value):
        return "normal"

    text = str(value).strip().lower()

    if text in {"0", "normal", "benign", "false", "none", "no_attack", "no attack"}:
        return "normal"
    if "dos" in text or "flood" in text or "frequency" in text:
        return "DoS"
    if "fuzz" in text or "random" in text or "malformed" in text:
        return "fuzzy"
    if "replay" in text:
        return "replay"
    if "spoof" in text or "imperson" in text or "masquerade" in text or "payload" in text or "dlc" in text or "timing" in text:
        return "spoofing"
    if text.isdigit() and text != "0":
        return "attack"

    return text


def prediction_to_binary_label(value: Any) -> str:
    return "normal" if value == "normal" else "attack"


def evaluate_detection_results(
    y_true: Sequence[Any],
    y_pred: Sequence[Any],
    *,
    mode: str = "binary",
) -> Dict[str, Any]:
    """
    Compute and print evaluation metrics.
    """
    if mode not in {"binary", "multiclass"}:
        raise ValueError("mode must be either 'binary' or 'multiclass'.")

    if mode == "binary":
        y_true_eval = pd.Series(y_true).map(normalize_binary_label)
        y_pred_eval = pd.Series(y_pred).map(prediction_to_binary_label)
        labels = ["normal", "attack"]
        pos_label = "attack"
    else:
        y_true_eval = pd.Series(y_true).map(normalize_multiclass_label)
        y_pred_eval = pd.Series(y_pred).map(normalize_multiclass_label)
        labels = sorted(set(y_true_eval.dropna().unique()) | set(y_pred_eval.dropna().unique()))
        pos_label = None

    accuracy = accuracy_score(y_true_eval, y_pred_eval)

    if mode == "binary":
        precision = precision_score(y_true_eval, y_pred_eval, pos_label=pos_label, zero_division=0)
        recall = recall_score(y_true_eval, y_pred_eval, pos_label=pos_label, zero_division=0)
        f1 = f1_score(y_true_eval, y_pred_eval, pos_label=pos_label, zero_division=0)
    else:
        precision = precision_score(y_true_eval, y_pred_eval, average="weighted", zero_division=0)
        recall = recall_score(y_true_eval, y_pred_eval, average="weighted", zero_division=0)
        f1 = f1_score(y_true_eval, y_pred_eval, average="weighted", zero_division=0)

    cm = confusion_matrix(y_true_eval, y_pred_eval, labels=labels)
    report = classification_report(y_true_eval, y_pred_eval, labels=labels, zero_division=0)

    print(f"\n{mode.capitalize()} evaluation")
    print(f"Accuracy : {accuracy:.6f}")
    print(f"Precision: {precision:.6f}")
    print(f"Recall   : {recall:.6f}")
    print(f"F1-score : {f1:.6f}")
    print("\nConfusion matrix:")
    print(pd.DataFrame(cm, index=[f"true_{x}" for x in labels], columns=[f"pred_{x}" for x in labels]))
    print("\nClassification report:")
    print(report)

    return {
        "mode": mode,
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "labels": labels,
        "confusion_matrix": cm.tolist(),
        "classification_report": report,
    }


## Pipeline Functions


In [59]:
# ============================================================
# PIPELINE FUNCTIONS
# ============================================================

def _feature_config_from_globals() -> Dict[str, Any]:
    return {
        "max_payload_bytes": int(MAX_PAYLOAD_BYTES),
        "rolling_packet_window": int(ROLLING_PACKET_WINDOW),
        "time_window_seconds": float(TIME_WINDOW_SECONDS),
    }


def _feature_config_from_detector(detector: CANRuleBasedDetector) -> Dict[str, Any]:
    saved = dict(detector.feature_config or {})
    return {
        "max_payload_bytes": int(saved.get("max_payload_bytes", MAX_PAYLOAD_BYTES)),
        "rolling_packet_window": int(saved.get("rolling_packet_window", ROLLING_PACKET_WINDOW)),
        "time_window_seconds": float(saved.get("time_window_seconds", TIME_WINDOW_SECONDS)),
    }


def print_detector_summary(detector: CANRuleBasedDetector) -> None:
    """
    Print calibrated thresholds and learned benign profiles.
    """
    print("\n" + "=" * 80)
    print("CALIBRATED RULE DETECTOR SUMMARY")
    print("=" * 80)
    print(f"Known benign CAN IDs learned: {len(detector.allowed_ids)}")
    print(f"Allowed benign ID transitions learned: {len(detector.allowed_transitions)}")
    print(f"High quantile: {detector.high_quantile}")
    print(f"Low quantile : {detector.low_quantile}")
    print(f"Minimum attack score: {detector.min_attack_score}")
    print(f"Core features used: {len(CORE_FEATURE_COLUMNS)}")

    if detector.feature_config:
        print("\nFeature settings saved in profile:")
        for key, value in detector.feature_config.items():
            print(f"  {key}: {value}")

    if PRINT_FULL_THRESHOLD_TABLE:
        print("\nGlobal thresholds:")
        for key, value in detector.global_thresholds.items():
            print(f"  {key}: {value}")

        print("\nFirst 20 learned benign CAN IDs:")
        print(sorted(int(x) for x in detector.allowed_ids)[:20])


def print_prediction_summary(predictions_df: pd.DataFrame) -> None:
    """
    Print compact final prediction results.
    """
    print("\nPrediction distribution:")
    print(predictions_df["predicted_label"].value_counts(dropna=False).to_string())

    score_cols = [col for col in predictions_df.columns if col.startswith("score_")]
    if score_cols:
        print("\nAverage rule score by class:")
        print(predictions_df[score_cols].mean().sort_values(ascending=False).to_string())

    rule_cols = [col for col in predictions_df.columns if col.startswith("rule_")]
    if rule_cols:
        top_rules = predictions_df[rule_cols].sum().sort_values(ascending=False).head(20)
        print("\nTop triggered rules:")
        print(top_rules.to_string())

    display_cols = [col for col in PRINT_PREDICTION_COLUMNS if col in predictions_df.columns]
    if display_cols:
        suspicious = predictions_df.sort_values("max_rule_score", ascending=False).head(PRINT_TOP_SUSPICIOUS_ROWS)
        print(f"\nTop {PRINT_TOP_SUSPICIOUS_ROWS} most suspicious rows:")
        print(suspicious[display_cols].to_string(index=False))


def train_detector_and_save_profile(
    train_file: str | Path,
    profile_file: str | Path,
    config: CANColumnConfig,
    *,
    max_payload_bytes: int = 8,
    rolling_packet_window: int = 100,
    time_window_seconds: float = 1.0,
    high_quantile: float = 0.995,
    low_quantile: float = 0.005,
    min_attack_score: float = 2.0,
) -> Tuple[CANRuleBasedDetector, pd.DataFrame]:
    """
    Load benign training data, extract compact features, fit thresholds,
    and save the calibrated detector profile to disk.
    """
    print("\n" + "=" * 80)
    print("TRAINING / CALIBRATION")
    print("=" * 80)
    print(f"Reading benign training file: {train_file}")

    train_raw_df = load_can_csv(train_file)
    train_features_df, _ = extract_features_for_rule_detector(
        train_raw_df,
        config,
        max_payload_bytes=max_payload_bytes,
        rolling_packet_window=rolling_packet_window,
        time_window_seconds=time_window_seconds,
        print_summary=False,
    )

    detector = CANRuleBasedDetector(
        high_quantile=high_quantile,
        low_quantile=low_quantile,
        min_attack_score=min_attack_score,
    ).fit(train_features_df)

    feature_config = {
        "max_payload_bytes": int(max_payload_bytes),
        "rolling_packet_window": int(rolling_packet_window),
        "time_window_seconds": float(time_window_seconds),
    }
    detector.save(profile_file, feature_config=feature_config)

    print("Rule-based detector calibrated and saved.")
    print(f"Profile file: {profile_file}")
    print(f"Training rows: {len(train_features_df)}")
    print(f"Training columns after preprocessing/features: {len(train_features_df.columns)}")
    print_detector_summary(detector)

    return detector, train_features_df


def evaluate_files(
    evaluation_files: Sequence[str | Path],
    detector: CANRuleBasedDetector,
    config: CANColumnConfig,
    *,
    evaluation_mode: str = "binary",
) -> pd.DataFrame:
    """
    Read evaluation files, extract compact features using the saved training
    feature settings, predict labels, and compute metrics when true labels exist.
    """
    feature_config = _feature_config_from_detector(detector)
    all_results: List[pd.DataFrame] = []

    for file_path in evaluation_files:
        file_path = Path(file_path)
        print("\n" + "=" * 80)
        print(f"EVALUATING FILE: {file_path}")
        print("=" * 80)

        raw_eval_df = load_can_csv(file_path)
        eval_features_df, true_labels = extract_features_for_rule_detector(
            raw_eval_df,
            config,
            max_payload_bytes=feature_config["max_payload_bytes"],
            rolling_packet_window=feature_config["rolling_packet_window"],
            time_window_seconds=feature_config["time_window_seconds"],
            print_summary=False,
        )

        predictions_df = detector.predict(eval_features_df, return_scores=True)
        predictions_df["source_file"] = file_path.name

        if true_labels is not None:
            predictions_df["true_label"] = true_labels.values
            evaluate_detection_results(
                predictions_df["true_label"],
                predictions_df["predicted_label"],
                mode=evaluation_mode,
            )
        else:
            print("No true label column found. Predictions generated without evaluation metrics.")

        print_prediction_summary(predictions_df)
        all_results.append(predictions_df)

    combined_results = pd.concat(all_results, ignore_index=True) if all_results else pd.DataFrame()

    if len(evaluation_files) > 1 and not combined_results.empty:
        print("\n" + "=" * 80)
        print("COMBINED FINAL RESULTS")
        print("=" * 80)
        print_prediction_summary(combined_results)

        if "true_label" in combined_results.columns:
            evaluate_detection_results(
                combined_results["true_label"],
                combined_results["predicted_label"],
                mode=evaluation_mode,
            )

    return combined_results


def evaluate_files_from_saved_profile(
    evaluation_files: Sequence[str | Path],
    profile_file: str | Path,
    config: CANColumnConfig,
    *,
    evaluation_mode: str = "binary",
) -> pd.DataFrame:
    """
    Load a previously saved detector profile and evaluate files without training.
    """
    profile_file = Path(profile_file)
    if not profile_file.exists():
        raise FileNotFoundError(
            f"Detector profile not found: {profile_file}. "
            "Run the training stage once to create it."
        )

    detector = CANRuleBasedDetector.load(profile_file)
    print(f"Loaded detector profile: {profile_file}")
    print_detector_summary(detector)

    return evaluate_files(
        evaluation_files,
        detector,
        config,
        evaluation_mode=evaluation_mode,
    )


## Training Stage — Run Only When Updating the Profile


In [60]:
# ============================================================
# TRAINING STAGE
# ============================================================
# Set RUN_TRAINING_STAGE = True in the configuration cell, run this cell once,
# then set it back to False for repeated evaluation-only runs.

if RUN_TRAINING_STAGE:
    config = build_column_config()
    detector, train_features_df = train_detector_and_save_profile(
        TRAINING_FILE,
        DETECTOR_PROFILE_FILE,
        config,
        max_payload_bytes=MAX_PAYLOAD_BYTES,
        rolling_packet_window=ROLLING_PACKET_WINDOW,
        time_window_seconds=TIME_WINDOW_SECONDS,
        high_quantile=HIGH_QUANTILE,
        low_quantile=LOW_QUANTILE,
        min_attack_score=MIN_ATTACK_SCORE,
    )
else:
    print("Training stage skipped. Set RUN_TRAINING_STAGE = True to create/update the saved profile.")


Training stage skipped. Set RUN_TRAINING_STAGE = True to create/update the saved profile.


## Evaluation Stage — Re-run This Cell with Different Files


In [61]:
# ============================================================
# EVALUATION STAGE
# ============================================================
# This cell loads DETECTOR_PROFILE_FILE and does not recalculate training thresholds.
# Edit EVALUATION_FILES in the configuration cell, then rerun this cell.

if RUN_EVALUATION_STAGE:
    if Path(DETECTOR_PROFILE_FILE).exists():
        config = build_column_config()
        results_df = evaluate_files_from_saved_profile(
            EVALUATION_FILES,
            DETECTOR_PROFILE_FILE,
            config,
            evaluation_mode=EVALUATION_MODE,
        )
    else:
        print(
            f"Detector profile not found: {DETECTOR_PROFILE_FILE}. "
            "Set RUN_TRAINING_STAGE = True and run the training stage once to create it."
        )
else:
    print("Evaluation stage skipped. Set RUN_EVALUATION_STAGE = True to evaluate files.")


Loaded detector profile: can_detector_profile.json

CALIBRATED RULE DETECTOR SUMMARY
Known benign CAN IDs learned: 52
Allowed benign ID transitions learned: 1180
High quantile: 0.995
Low quantile : 0.005
Minimum attack score: 2.0
Core features used: 11

Feature settings saved in profile:
  max_payload_bytes: 8
  rolling_packet_window: 100
  time_window_seconds: 1.0

Global thresholds:
  consecutive_id_streak_high: 5.0
  delta_t_global_low: 9.799003601074219e-05
  hamming_same_id_high: 25.0
  packet_rate_high: 1617.0
  payload_entropy_high: 3.0
  rolling_id_entropy_high: 5.035134314233857
  rolling_unique_id_count_high: 40.0
  same_id_rate_high: 101.0

First 20 learned benign CAN IDs:
[27, 89, 99, 142, 159, 161, 162, 169, 179, 233, 234, 235, 236, 237, 243, 275, 299, 346, 349, 363]

EVALUATING FILE: eval3.csv

Binary evaluation
Accuracy : 0.688743
Precision: 0.067873
Recall   : 1.000000
F1-score : 0.127119

Confusion matrix:
             pred_normal  pred_attack
true_normal         2645 

In [62]:
from pathlib import Path

def extract_print_and_save_features_from_csv(
    evaluation_csv_path,
    output_csv_path="extracted_features.csv",
    profile_file=DETECTOR_PROFILE_FILE,
    print_rows=20,
):
    """
    Reuse the notebook's existing feature-extraction pipeline.

    This function:
    1. Loads an evaluation CSV file
    2. Extracts the same simplified features used by the detector
    3. Prints the extracted features
    4. Saves them to a CSV file
    """

    evaluation_csv_path = Path(evaluation_csv_path)
    output_csv_path = Path(output_csv_path)

    config = build_column_config()

    # Use the same feature settings saved during training, if available.
    # This avoids mismatches between training and evaluation feature extraction.
    if Path(profile_file).exists():
        detector = CANRuleBasedDetector.load(profile_file)
        feature_config = _feature_config_from_detector(detector)
        print(f"Loaded feature settings from saved profile: {profile_file}")
    else:
        feature_config = _feature_config_from_globals()
        print("Saved detector profile not found.")
        print("Using current notebook global feature settings instead.")

    # Load raw CAN CSV
    raw_eval_df = load_can_csv(evaluation_csv_path)

    # Reuse the existing feature extraction function
    features_df, true_labels = extract_features_for_rule_detector(
        raw_eval_df,
        config,
        max_payload_bytes=feature_config["max_payload_bytes"],
        rolling_packet_window=feature_config["rolling_packet_window"],
        time_window_seconds=feature_config["time_window_seconds"],
        print_summary=False,
    )

    # Optional: keep the label column if the evaluation CSV has one
    if true_labels is not None:
        features_df["true_label"] = true_labels.values

    # Print summary
    print("\n" + "=" * 80)
    print("FEATURE EXTRACTION COMPLETED")
    print("=" * 80)
    print(f"Input file : {evaluation_csv_path}")
    print(f"Output file: {output_csv_path}")
    print(f"Rows       : {len(features_df)}")
    print(f"Columns    : {len(features_df.columns)}")

    print("\nExtracted feature columns:")
    for col in features_df.columns:
        print(f"- {col}")

    print(f"\nFirst {print_rows} extracted rows:")
    print(features_df.head(print_rows).to_string(index=False))

    # Save to CSV
    features_df.to_csv(output_csv_path, index=False)

    print(f"\nSaved extracted features to: {output_csv_path}")

    return features_df

In [64]:
features_df = extract_print_and_save_features_from_csv(
    evaluation_csv_path="eval2.csv",
    output_csv_path="eval2_extracted_features.csv"
)

Loaded feature settings from saved profile: can_detector_profile.json

FEATURE EXTRACTION COMPLETED
Input file : eval2.csv
Output file: eval2_extracted_features.csv
Rows       : 9971
Columns    : 27

Extracted feature columns:
- arbitration_id
- timestamp_raw
- timestamp_seconds
- timestamp_parse_valid
- time_from_start_s
- arbitration_id_raw
- arbitration_id_parse_valid
- arbitration_id_hex
- data_field_raw
- payload_hex_normalized
- payload_is_valid_hex
- payload_is_missing
- payload_has_odd_nibble_count
- dlc
- payload_exceeds_configured_length
- delta_t_global_s
- delta_t_same_id_s
- packet_rate_last_time_window_hz
- same_id_rate_last_time_window_hz
- consecutive_id_streak_length
- prev_arbitration_id
- rolling_unique_id_count
- rolling_id_entropy_bits
- payload_byte_sum
- payload_byte_entropy_bits
- payload_hamming_distance_prev_same_id
- true_label

First 20 extracted rows:
 arbitration_id      timestamp_raw  timestamp_seconds  timestamp_parse_valid  time_from_start_s arbitration